# 09. 짝 라벨 후처리 규칙

제출용 노트북에 그대로 넣을 수 있는 자체 완결형 구현이다. `scripts/apply_pair_rule.py` 를
import 하지 않는다 — 제출한 노트북만으로 제출한 csv 가 재현돼야 하기 때문이다.

## 무엇을 하는가

test 행의 유전자 프로파일이 train 의 **유일한** 행과 바이트 단위로 완전히 같고 그 train 라벨이
`KIPAN`·`KIRC`·`GBMLGG`·`LGG` 중 하나면, 예측을 매칭된 라벨이 아니라 **짝 코호트 라벨**로 바꾼다.

TCGA 에서 `KIPAN = KICH ∪ KIRC ∪ KIRP`, `GBMLGG = GBM ∪ LGG` 이므로 같은 환자가 상위·하위
코호트 라벨로 두 번 들어가 있다. 그 짝 중 한쪽만 train 에 있는 행의 반대편이 test 로 갔다.
따라서 평범한 1-NN 과 **반대 방향**으로 라벨을 낸다.

## 규정

대회가 금지하는 것은 *모델 학습에서* 평가 데이터셋을 활용하는 것이다(인코더·스케일러를 test 로
fit 하거나 test 결측치를 test 통계로 메우는 것). 이 규칙은 학습에 test 를 넣지 않는다.

- **짝 매핑도 임계값도 `train.csv` 에서만 유도한다.** 아래 §2·§3 이 그 유도 과정이고,
  하드코딩된 라벨 매핑이 없다.
- **행 단위로 독립이다.** 규칙의 진입점 `PairRule.apply_to_row(profile)` 는 프로파일 **한 개**만
  받는다. 다른 test 행을 볼 수 없다는 게 시그니처에 드러난다.
- test 두 행끼리 프로파일을 맞춰 보는 방식은 **쓰지 않는다.** 행 단위 독립이 깨진다.

## 주의

이 규칙은 로컬로 검증할 수 없다. group CV 는 같은 프로파일을 한 fold 로 묶어 이 상황 자체를
못 만들고, 짝 안에서의 스왑이라 예측 분포도 거의 안 움직인다. 맞으면 macro F1 +0.05~0.08,
거꾸로면 비슷한 크기로 떨어진다.

In [1]:
from __future__ import annotations

import hashlib
import random
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd


def find_raw() -> Path:
    """노트북 위치가 바뀌어도 원본 csv 를 찾는다."""
    for cand in (Path("data/raw"), Path("../data/raw"), Path("code/data/raw"), Path("../code/data/raw")):
        if (cand / "train.csv").exists():
            return cand.resolve()
    raise FileNotFoundError("train.csv 를 못 찾았다. RAW 를 직접 지정할 것.")


RAW = find_raw()

#: 매칭에 요구하는 최소 변이 수. 근거는 §2 에서 train 만으로 재계산한다.
MIN_MUT = 3

print(RAW)

D:\Code\Final_Hachathon\code\data\raw


## 1. 원본 csv 를 바이트로 읽는다

pandas 로 읽지 않는다. 빈 셀이 `NaN` 이 되고 dtype 추론이 끼어들면 "완전히 같은가" 판정이
흔들린다. 원문 바이트를 그대로 들고 비교한다.

그리고 **train 과 test 의 유전자 컬럼 순서가 같은지 먼저 확인한다.** 이게 어긋나 있으면
바이트 비교 자체가 무의미해진다.

In [2]:
def read_rows(path: Path, *, labelled: bool) -> tuple[list[bytes], list[tuple[str, str | None, bytes]]]:
    """`(유전자 헤더, [(id, label, profile_bytes), ...])` 를 낸다."""
    rows: list[tuple[str, str | None, bytes]] = []
    with path.open("rb") as fh:
        header = fh.readline().rstrip(b"\r\n").split(b",")
        genes = header[2:] if labelled else header[1:]
        for line in fh:
            line = line.rstrip(b"\r\n")
            if not line:
                continue
            i = line.find(b",")
            if labelled:
                j = line.find(b",", i + 1)
                rows.append((line[:i].decode(), line[i + 1 : j].decode(), line[j + 1 :]))
            else:
                rows.append((line[:i].decode(), None, line[i + 1 :]))
    return genes, rows


train_genes, train_rows = read_rows(RAW / "train.csv", labelled=True)
test_genes, test_rows = read_rows(RAW / "test.csv", labelled=False)

assert train_genes == test_genes, "train/test 유전자 컬럼이 다르다 — 바이트 비교가 성립하지 않는다"
print(f"train {len(train_rows)}행 · test {len(test_rows)}행 · 유전자 {len(train_genes)}열")

train 6201행 · test 2546행 · 유전자 4384열


In [3]:
def n_mut(profile: bytes) -> int:
    """변이 셀 수. 빈 셀은 WT 로 본다(test 에 237개 있다)."""
    return sum(cell not in (b"WT", b"") for cell in profile.split(b","))


def digest(profile: bytes) -> bytes:
    return hashlib.blake2b(profile, digest_size=16).digest()


#: 프로파일 해시 -> [(id, label, profile), ...]. train 에서만 만든다.
train_index: dict[bytes, list[tuple[str, str, bytes]]] = defaultdict(list)
for sid, label, profile in train_rows:
    train_index[digest(profile)].append((sid, label, profile))

print(f"고유 프로파일 {len(train_index)}종 · 중복 묶음 {sum(1 for v in train_index.values() if len(v) > 1)}개")

고유 프로파일 5636종 · 중복 묶음 451개


## 2. 근거 ①: train 내부 중복 구조

프로파일이 완전히 같은 train 행 묶음을 변이 수 구간별로 갈라 본다. 두 가지를 동시에 읽는다.

1. **짝 구조가 실재하는가** — 같은 라벨 묶음이 있는지.
2. **임계값을 어디에 둘 것인가** — 우연 일치가 어느 구간에서 끝나는지.

test 를 보지 않고 둘 다 답한다.

In [4]:
def bucket(n: int) -> str:
    if n <= 2:
        return str(n)
    return "3~5" if n <= 5 else ("6~10" if n <= 10 else "11+")


ORDER = ["0", "1", "2", "3~5", "6~10", "11+"]
COLS = ["묶음", "같은 라벨", "3행 이상", "깨끗한 짝"]
agg: dict[str, Counter] = defaultdict(Counter)

# 뒤의 세 열은 배타적 분할이 아니라 각각 독립으로 세는 지표다.
# (같은 라벨이면서 3행 이상인 묶음은 두 열에 모두 잡힌다.)
for rs in train_index.values():
    if len(rs) < 2:
        continue
    b = agg[bucket(n_mut(rs[0][2]))]
    b["묶음"] += 1
    if len({r[1] for r in rs}) == 1:
        b["같은 라벨"] += 1                    # 우연 일치의 흔적
    if len(rs) > 2:
        b["3행 이상"] += 1                    # 짝 구조로 설명이 안 되는 묶음
    if len(rs) == 2 and rs[0][1] != rs[1][1]:
        b["깨끗한 짝"] += 1                    # 규칙이 쓰는 형태

dup_table = pd.DataFrame(
    [{"변이 수": b, **{c: agg[b][c] for c in COLS}} for b in ORDER if b in agg]
).set_index("변이 수")
dup_table

,묶음,같은 라벨,3행 이상,깨끗한 짝
변이 수,,,,
0,1,0,1,0
1,21,4,5,14
2,7,0,0,7
3~5,64,0,0,64
6~10,135,0,0,135
11+,223,0,0,223


우연 일치는 **변이 1개 구간에만** 실재한다. 거기서만 같은 라벨 묶음과 3행 이상 묶음이 나오고,
변이 2개부터는 전부 깨끗한 2행 묶음이다. `MIN_MUT = 3` 은 그 경계보다 한 칸 보수적으로 잡은 값이다.

아래에서 임계값별로 다시 확인한다.

In [5]:
for t in (1, 2, 3, 6):
    sel = [rs for rs in train_index.values() if len(rs) > 1 and n_mut(rs[0][2]) >= t]
    same = sum(1 for rs in sel if len({r[1] for r in rs}) == 1)
    big = sum(1 for rs in sel if len(rs) > 2)
    flag = "깨끗" if same == 0 and big == 0 else "예외 있음"
    print(f"min_mut >= {t}: 묶음 {len(sel):>3} · 같은 라벨 {same} · 3행 이상 {big}  [{flag}]")

min_mut >= 1: 묶음 450 · 같은 라벨 4 · 3행 이상 5  [예외 있음]
min_mut >= 2: 묶음 429 · 같은 라벨 0 · 3행 이상 0  [깨끗]


min_mut >= 3: 묶음 422 · 같은 라벨 0 · 3행 이상 0  [깨끗]
min_mut >= 6: 묶음 358 · 같은 라벨 0 · 3행 이상 0  [깨끗]


## 3. 짝 매핑을 train 에서 유도한다

라벨 매핑을 손으로 적지 않는다. train 중복 묶음이 어떤 라벨끼리 붙어 있는지 세어서 만든다.
그래야 "test 를 보고 정한 것 아니냐" 는 물음에 코드 자체가 답한다.

한 라벨이 두 개 이상의 짝에 속하면 매핑이 모호해지므로 그 경우는 예외를 던진다.

In [6]:
def derive_pair_map(index: dict, min_mut: int) -> dict[str, str]:
    """train 중복 묶음에서 짝 매핑을 유도한다. test 를 인자로 받지 않는다."""
    pairs: set[frozenset[str]] = set()
    for rs in index.values():
        if len(rs) == 2 and n_mut(rs[0][2]) >= min_mut and rs[0][1] != rs[1][1]:
            pairs.add(frozenset((rs[0][1], rs[1][1])))

    members = Counter(label for p in pairs for label in p)
    ambiguous = [label for label, c in members.items() if c > 1]
    if ambiguous:
        raise ValueError(f"한 라벨이 여러 짝에 속한다 — 매핑이 모호하다: {ambiguous}")

    mapping: dict[str, str] = {}
    for p in pairs:
        a, b = sorted(p)
        mapping[a], mapping[b] = b, a
    return mapping


PAIR = derive_pair_map(train_index, MIN_MUT)

assert all(PAIR[PAIR[k]] == k for k in PAIR), "짝 매핑이 대칭이 아니다"
assert set(PAIR) == {"KIPAN", "KIRC", "GBMLGG", "LGG"}, f"예상 밖의 짝: {set(PAIR)}"
PAIR

{'KIPAN': 'KIRC', 'KIRC': 'KIPAN', 'GBMLGG': 'LGG', 'LGG': 'GBMLGG'}

## 4. 근거 ②: 고아 프로파일

짝이 train 안에 없는 행(고아)을 라벨별로 센다. 이 수가 §6 의 test 매칭 수와 맞는 것이
규칙의 결정적 근거다 — 짝의 반대편이 test 로 갔다는 뜻이다.

여기서는 train 만 센다. 대조는 규칙을 적용한 뒤에 한다.

In [7]:
orphan: Counter = Counter()
paired: Counter = Counter()
for rs in train_index.values():
    if n_mut(rs[0][2]) < MIN_MUT:
        continue
    for _, label, _ in rs:
        if label in PAIR:
            (orphan if len(rs) == 1 else paired)[label] += 1

pd.DataFrame(
    {"train 안에 짝 있음": paired, "짝이 train 밖(고아)": orphan}
).fillna(0).astype(int).loc[sorted(PAIR)]

,train 안에 짝 있음,짝이 train 밖(고아)
GBMLGG,168,280
KIPAN,254,233
KIRC,254,57
LGG,168,50


## 5. 규칙 — 행 하나만 받는다

`apply_to_row` 가 프로파일 **한 개**만 받는다. test 전체를 넘길 방법이 없으므로 다른 test 행의
영향을 받는 게 구조적으로 불가능하다. "test 한 행만 따로 넣어도 같은 결과가 나오는가" 판정을
코드 모양으로 만족시킨다.

규칙 대상이 아니면 `None` 을 내고, 그 행의 예측은 모델이 낸 값을 그대로 둔다.

In [8]:
class PairRule:
    """짝 라벨로 변환하는 exact-match 1-NN.

    train 에서 만든 조회 테이블과 짝 매핑만 들고 있는다. test 는 생성자에도 들어오지 않는다.
    """

    def __init__(self, index: dict, pair: dict[str, str], min_mut: int) -> None:
        self._index = index
        self._pair = pair
        self._min_mut = min_mut

    def apply_to_row(self, profile: bytes) -> str | None:
        """짝 라벨을 내거나, 규칙 대상이 아니면 None 을 낸다."""
        if n_mut(profile) < self._min_mut:
            return None                       # 우연 일치 위험 구간
        hits = [r for r in self._index.get(digest(profile), ()) if r[2] == profile]
        if len(hits) != 1:
            return None                       # 매칭 없음, 또는 train 에 이미 양쪽이 다 있음
        return self._pair.get(hits[0][1])     # 짝 4종이 아니면 None


rule = PairRule(train_index, PAIR, MIN_MUT)

## 6. 적용

In [9]:
flips: dict[str, str] = {}
for sid, _, profile in test_rows:
    label = rule.apply_to_row(profile)
    if label is not None:
        flips[sid] = label

matched = Counter(PAIR[v] for v in flips.values())   # 매칭된 train 라벨 = 짝의 반대
summary = pd.DataFrame(
    {"고아 수(train)": orphan, "test 매칭 수": matched}
).fillna(0).astype(int).loc[sorted(PAIR)]

print(f"규칙 대상 {len(flips)}행 / test {len(test_rows)}행 ({len(flips) / len(test_rows):.2%})\n")
summary

규칙 대상 214행 / test 2546행 (8.41%)



,고아 수(train),test 매칭 수
GBMLGG,280,48
KIPAN,233,59
KIRC,57,57
LGG,50,50


`KIRC` 와 `LGG` 행에서 고아 수와 test 매칭 수가 정확히 맞으면 근거 ②가 성립한다 —
train 에서 짝을 못 찾은 행의 반대편이 전부 test 에 있다는 뜻이다.

`KIPAN`·`GBMLGG` 는 상위 코호트라 다른 하위 암종(KICH·KIRP·GBM) 몫이 고아에 섞여 있어
수가 맞지 않는 게 정상이다.

## 7. 검증

이 규칙은 CV 로 잴 수 없다. 구현이 조용히 틀리면 제출 한 번을 태우기 전까지 아무도 모른다.
그래서 전제를 전부 `assert` 로 박아 둔다. 하나라도 깨지면 제출하지 않는다.

In [10]:
# ① 프로파일이 같으면 예외 없이 라벨이 갈리고, 묶음 크기는 전부 2다.
groups = [rs for rs in train_index.values() if len(rs) > 1 and n_mut(rs[0][2]) >= MIN_MUT]
assert all(len(rs) == 2 for rs in groups), "3행 이상 묶음이 있다"
assert all(rs[0][1] != rs[1][1] for rs in groups), "같은 라벨 묶음이 있다"
assert all(PAIR.get(rs[0][1]) == rs[1][1] for rs in groups), "짝이 아닌 묶음이 있다"

# ② 고아 수와 test 매칭 수가 하위 코호트에서 일치한다.
for label in ("KIRC", "LGG"):
    assert orphan[label] == matched[label], f"{label}: 고아 {orphan[label]} != 매칭 {matched[label]}"

# ③ 바뀌는 라벨은 짝 4종뿐이다.
assert set(flips.values()) <= set(PAIR)

print(f"전제 확인 통과 · 중복 묶음 {len(groups)}개 · 규칙 대상 {len(flips)}행")

전제 확인 통과 · 중복 묶음 422개 · 규칙 대상 214행


In [11]:
# ④ 행 단위 독립 — 순서를 섞어도 결과가 같다.
#    apply_to_row 가 행 하나만 받으므로 구조적으로 보장되지만, 리팩터링 회귀를 막는 가드로 둔다.
shuffled = test_rows[:]
random.Random(0).shuffle(shuffled)
reshuffled = {sid: lab for sid, _, p in shuffled if (lab := rule.apply_to_row(p)) is not None}
assert reshuffled == flips, "test 행 순서가 결과를 바꾼다 — 행 단위 독립이 깨졌다"

# ⑤ 한 행만 따로 넣어도 같은 답이 나온다.
for sid, _, profile in test_rows:
    solo = PairRule(train_index, PAIR, MIN_MUT).apply_to_row(profile)
    assert solo == flips.get(sid), f"{sid}: 단독 입력 결과가 다르다"

print(f"행 단위 독립 확인 통과 — {len(test_rows)}행 전부 단독 입력과 전체 입력의 결과가 같다")

행 단위 독립 확인 통과 — 2546행 전부 단독 입력과 전체 입력의 결과가 같다


## 8. 제출 파일에 적용

모델이 낸 예측을 `ID`·`SUBCLASS` 두 컬럼 DataFrame 으로 넘기면 된다. 같은 노트북 안에서
학습까지 하는 경우에도 `apply_to_submission(sub)` 한 줄이면 붙는다.

In [12]:
def apply_to_submission(sub: pd.DataFrame) -> pd.DataFrame:
    if list(sub.columns) != ["ID", "SUBCLASS"]:
        raise ValueError(f"컬럼이 ['ID', 'SUBCLASS'] 가 아니다: {list(sub.columns)}")
    out = sub.copy()
    out["SUBCLASS"] = [flips.get(i, c) for i, c in zip(out["ID"], out["SUBCLASS"])]
    return out


# 입력 제출 파일을 여기서 지정한다. 노트북 안에서 학습한 경우 그 예측 DataFrame 을 바로 넘기면 된다.
# 기본값은 지금까지 LB 로 확인된 유일한 구성(v002, CV 0.5165 / LB 0.3896)이다.
SUBMISSION_IN = Path("../../Models/Ensemble/v002_seed42_f16_group5_macroF1_0.5165/submission.csv")
SUBMISSION_OUT = Path("../artifacts/submissions/submission_notebook_pairrule_m3.csv")

base = pd.read_csv(SUBMISSION_IN)
final = apply_to_submission(base)

changed = int((final["SUBCLASS"].to_numpy() != base["SUBCLASS"].to_numpy()).sum())
copying = sum(1 for i, c in zip(base["ID"], base["SUBCLASS"]) if i in flips and PAIR.get(c) == flips[i])
print(f"규칙 대상 {len(flips)}행 · 실제로 바뀐 행 {changed} · 바꾸기 전 train 라벨을 복사하던 행 {copying}")

규칙 대상 214행 · 실제로 바뀐 행 214 · 바꾸기 전 train 라벨을 복사하던 행 213


In [13]:
# 제출 스키마 확인 후 저장
sample = pd.read_csv(RAW / "sample_submission.csv")
assert list(final.columns) == ["ID", "SUBCLASS"]
assert len(final) == len(sample)
assert (final["ID"].to_numpy() == sample["ID"].to_numpy()).all(), "ID 순서가 sample 과 다르다"
assert final["SUBCLASS"].notna().all()

SUBMISSION_OUT.parent.mkdir(parents=True, exist_ok=True)
final.to_csv(SUBMISSION_OUT, index=False, encoding="UTF-8-sig")
print(f"→ {SUBMISSION_OUT}")

→ ..\artifacts\submissions\submission_notebook_pairrule_m3.csv


## 9. 기존 스크립트와 대조 (선택)

`scripts/apply_pair_rule.py` 로 이미 만들어 둔 후보가 있으면 노트북 결과와 같은지 본다.
제출한 노트북이 제출한 csv 를 재현하는지 확인하는 자리다.

In [14]:
REFERENCE = Path("../../Models/pairrule_candidates/submission_ens_v002_pairrule_m3.csv")

if REFERENCE.exists():
    ref = pd.read_csv(REFERENCE)
    merged = ref.merge(final, on="ID", suffixes=("_script", "_notebook"))
    assert len(merged) == len(ref), "ID 가 서로 맞지 않는다"
    diff = int((merged["SUBCLASS_script"] != merged["SUBCLASS_notebook"]).sum())
    print(f"스크립트 산출물과 다른 행: {diff} / {len(merged)}")
    assert diff == 0, "노트북과 스크립트 결과가 다르다 — 둘 중 하나가 틀렸다"
else:
    print("대조 대상이 없다 — 건너뛴다")

스크립트 산출물과 다른 행: 0 / 2546
